In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

c:\Users\Manindra\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:160: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
db_path = '../database/financial_data.duckdb'
conn = duckdb.connect(db_path, read_only=True)

query = "select * from gold_ml_features where asset_symbol = 'BTC' and interval = '1h'"
df = conn.execute(query).df()
print(f"total rows loaded: {len(df)}")

total rows loaded: 52755


In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

In [4]:
asset = df['asset_symbol'].iloc[0]
interval = df['interval'].iloc[0]
print(f" {asset} ({interval})")

 BTC (1h)


In [5]:
low = df['log_returns'].quantile(0.01)
high = df['log_returns'].quantile(0.99)
df['log_returns_cleaned'] = df['log_returns'].clip(lower=low, upper=high)

In [6]:
features = ['volume', 'rsi_14', 'atr_14', 'log_returns_cleaned']

In [7]:
df['target'] = (df['log_returns'].shift(-1) > 0).astype(int)
df = df.dropna()
print("target variable created successfully!")
print(df['target'].value_counts())

target variable created successfully!
target
1    26704
0    26051
Name: count, dtype: int64


In [8]:
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [9]:
display(df[features].head())

,volume,rsi_14,atr_14,log_returns_cleaned
0,0.914817,2.859818,-1.051879,3.350995
1,0.106885,0.659469,-0.988325,-3.459182
2,-0.477445,0.695790,-0.983516,0.244595
3,-0.452776,0.371741,-0.975395,-1.585321
4,-0.516567,0.726694,-0.973779,2.340582


In [10]:
X = df[features]
y = df['target']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [12]:
print(f"Training set size: {len(X_train)} candles")
print(f"Testing set size: {len(X_test)} candles")

Training set size: 42204 candles
Testing set size: 10551 candles


In [13]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
mlflow.set_experiment("BTC_Baseline_Model")
with mlflow.start_run(run_name="First_Random_Forest"):
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("asset", "BTC")
    
    predictions = model.predict(X_test)
    
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    
    mlflow.sklearn.log_model(model, "random_forest_btc")
    
    print(f"successfully logged to mlflow!")
    print(f"accuracy: {acc:.4f}")

2026/04/09 21:46:02 INFO mlflow.tracking.fluent: Experiment with name 'BTC_Baseline_Model' does not exist. Creating a new experiment.
2026/04/09 21:46:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 21:46:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


successfully logged to mlflow!
accuracy: 0.4985
🏃 View run First_Random_Forest at: http://localhost:5000/#/experiments/5/runs/9568cfe89f0647afbd324615a5143604
🧪 View experiment at: http://localhost:5000/#/experiments/5


In [15]:
conn.close()